# Прогноз клиентского потока и потребности касс

Краткий рабочий ноутбук для ежедневного расчета. Подробное описание логики вынесено в `README_cashdesk_sarima_bootstrap.md`.

In [ ]:
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from prefect.blocks.system import Secret
from sqlalchemy import text
from statsmodels.tsa.statespace.sarimax import SARIMAX
from toolbox import oracle

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. Параметры запуска

In [ ]:
SOURCE_TABLE = "AIDA2.AIDA_TRS_DTM_CASHOP@aida"
HISTORY_MONTHS = 18
RUN_DATE = pd.Timestamp.today().normalize()
DATE_FROM = (RUN_DATE - pd.DateOffset(months=HISTORY_MONTHS)).date().isoformat()
EXCLUDE_DATE_RANGES = [("2025-09-01", "2025-11-01")]

FORECAST_DAYS = 30
INITIAL_TRAIN_DAYS = 365
BACKTEST_STEP_DAYS = 7
BOOTSTRAP_ITERATIONS = 1000

SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 7)

MIN_OBSERVATIONS = INITIAL_TRAIN_DAYS + FORECAST_DAYS
CASH_NEED_CLIP_LOWER = None
CASHDESK_FILTER = None

## 2. Загрузка данных из Oracle

In [ ]:
USERNAME_CDW = "sb_analytics"

PASSWORD_CDW = (await Secret.load("pass-sb-analytics")).get()
engine_cdw = oracle.create_engine_cdw(USERNAME_CDW, PASSWORD_CDW)

In [ ]:
query = f"""
select
    atdtmco_cashdesk_name,
    atdtmco_calday,
    atdtmco_saldo_turn,
    atdtmco_ns
from {SOURCE_TABLE}
where atdtmco_calday >= date '{DATE_FROM}'
"""

for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
    query += (
        f"\n  and not ("
        f"atdtmco_calday >= date '{exclude_start}' "
        f"and atdtmco_calday < date '{exclude_end}'"
        f")"
    )

if CASHDESK_FILTER:
    escaped_names = ", ".join(["'" + name.replace("'", "''") + "'" for name in CASHDESK_FILTER])
    query += f"\n  and atdtmco_cashdesk_name in ({escaped_names})"

with engine_cdw.connect() as conn:
    raw_df = pd.read_sql(text(query), conn)

raw_df.columns = raw_df.columns.str.lower()
raw_df["atdtmco_calday"] = pd.to_datetime(raw_df["atdtmco_calday"])

print(raw_df.shape)
raw_df.head()

## 3. Подготовка дневных рядов

In [ ]:
NS_NEED_SIGN = 1

required_columns = {
    "atdtmco_cashdesk_name",
    "atdtmco_calday",
    "atdtmco_saldo_turn",
    "atdtmco_ns",
}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"В данных не хватает колонок: {sorted(missing_columns)}")

daily_df = (
    raw_df
    .assign(calday=lambda df: df["atdtmco_calday"].dt.normalize())
    .groupby(["atdtmco_cashdesk_name", "calday"], as_index=False)
    .agg(
        saldo_turn=("atdtmco_saldo_turn", "sum"),
        ns_min=("atdtmco_ns", "min"),
    )
)

daily_df["cash_need"] = NS_NEED_SIGN * daily_df["ns_min"]
daily_df = daily_df.sort_values(["atdtmco_cashdesk_name", "calday"]).reset_index(drop=True)

print(daily_df.shape)
daily_df.head()

## 4. Вспомогательные функции

In [ ]:
@dataclass
class SarimaConfig:
    order: tuple[int, int, int]
    seasonal_order: tuple[int, int, int, int]
    maxiter: int = 200


SARIMA_CONFIG = SarimaConfig(
    order=SARIMA_ORDER,
    seasonal_order=SARIMA_SEASONAL_ORDER,
)


def make_regular_daily_series(
    cashdesk_df: pd.DataFrame,
    value_col: str,
    fill_method: str,
) -> pd.Series:
    series = (
        cashdesk_df
        .set_index("calday")[value_col]
        .sort_index()
        .asfreq("D")
        .astype(float)
    )

    if fill_method == "zero":
        return series.fillna(0.0)
    if fill_method == "ffill":
        return series.ffill().bfill()
    if fill_method == "interpolate":
        return series.interpolate(method="time").ffill().bfill()

    raise ValueError(f"Неизвестный fill_method: {fill_method}")


def fit_sarima(y: pd.Series, config: SarimaConfig):
    model = SARIMAX(
        y,
        order=config.order,
        seasonal_order=config.seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit(disp=False, maxiter=config.maxiter)


def sarima_point_forecast(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
) -> pd.DataFrame:
    result = fit_sarima(y, config)
    forecast_result = result.get_forecast(steps=steps)
    forecast_mean = forecast_result.predicted_mean
    conf_int = forecast_result.conf_int()

    output = pd.DataFrame(
        {
            "forecast_mean": forecast_mean.values,
            "forecast_lower_95": conf_int.iloc[:, 0].values,
            "forecast_upper_95": conf_int.iloc[:, 1].values,
        },
        index=forecast_mean.index,
    )
    output.index.name = "forecast_date"
    return output.reset_index()


def collect_backtest_error_blocks(
    y: pd.Series,
    steps: int,
    initial_train_days: int,
    step_days: int,
    config: SarimaConfig,
) -> np.ndarray:
    error_blocks = []
    max_train_end = len(y) - steps

    for train_end in range(initial_train_days, max_train_end + 1, step_days):
        train = y.iloc[:train_end]
        test = y.iloc[train_end:train_end + steps]

        if train.notna().sum() < initial_train_days or len(test) < steps:
            continue

        try:
            fitted_model = fit_sarima(train, config)
            forecast = fitted_model.get_forecast(steps=steps).predicted_mean
            error = test.values - forecast.values

            if np.isfinite(error).all():
                error_blocks.append(error)
        except Exception as exc:
            print(f"Backtest iteration failed at train_end={train_end}: {exc}")
            continue

    if not error_blocks:
        raise ValueError("Не удалось собрать ни одного блока ошибок backtest.")

    return np.vstack(error_blocks)


def forecast_with_error_bootstrap(
    y: pd.Series,
    steps: int,
    initial_train_days: int,
    step_days: int,
    bootstrap_iterations: int,
    config: SarimaConfig,
    clip_lower: Optional[float] = None,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    error_blocks = collect_backtest_error_blocks(
        y=y,
        steps=steps,
        initial_train_days=initial_train_days,
        step_days=step_days,
        config=config,
    )

    final_model = fit_sarima(y, config)
    forecast_result = final_model.get_forecast(steps=steps)
    forecast_mean = forecast_result.predicted_mean

    sampled_block_indexes = rng.integers(
        low=0,
        high=len(error_blocks),
        size=bootstrap_iterations,
    )
    sampled_errors = error_blocks[sampled_block_indexes]
    scenarios = forecast_mean.values.reshape(1, -1) + sampled_errors

    if clip_lower is not None:
        scenarios = np.maximum(scenarios, clip_lower)

    output = pd.DataFrame(
        {
            "forecast_mean": forecast_mean.values,
            "forecast_p50": np.quantile(scenarios, 0.50, axis=0),
            "forecast_p85": np.quantile(scenarios, 0.85, axis=0),
            "forecast_p95": np.quantile(scenarios, 0.95, axis=0),
            "error_blocks_used": len(error_blocks),
        },
        index=forecast_mean.index,
    )
    output.index.name = "forecast_date"
    return output.reset_index(), scenarios, error_blocks

## 5. Прогноз для одной кассы

In [ ]:
def forecast_single_cashdesk(
    cashdesk_name: str,
    cashdesk_df: pd.DataFrame,
    config: SarimaConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    saldo_series = make_regular_daily_series(
        cashdesk_df=cashdesk_df,
        value_col="saldo_turn",
        fill_method="zero",
    )
    need_series = make_regular_daily_series(
        cashdesk_df=cashdesk_df,
        value_col="cash_need",
        fill_method="ffill",
    )

    if len(saldo_series) < MIN_OBSERVATIONS:
        raise ValueError(
            f"Недостаточно истории для кассы {cashdesk_name}: "
            f"{len(saldo_series)} дней, нужно минимум {MIN_OBSERVATIONS}."
        )

    saldo_forecast = sarima_point_forecast(
        y=saldo_series,
        steps=FORECAST_DAYS,
        config=config,
    )
    saldo_forecast.insert(0, "cashdesk_name", cashdesk_name)
    saldo_forecast.insert(1, "target", "saldo_turn")

    need_forecast, need_scenarios, need_error_blocks = forecast_with_error_bootstrap(
        y=need_series,
        steps=FORECAST_DAYS,
        initial_train_days=INITIAL_TRAIN_DAYS,
        step_days=BACKTEST_STEP_DAYS,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
        config=config,
        clip_lower=CASH_NEED_CLIP_LOWER,
    )
    need_forecast.insert(0, "cashdesk_name", cashdesk_name)
    need_forecast.insert(1, "target", "cash_need_min_ns")

    return saldo_forecast, need_forecast, need_scenarios, need_error_blocks

## 6. Проверка на одной кассе

In [ ]:
cashdesk_stats = (
    daily_df
    .groupby("atdtmco_cashdesk_name")
    .agg(
        date_min=("calday", "min"),
        date_max=("calday", "max"),
        days_with_rows=("calday", "nunique"),
        saldo_turn_mean=("saldo_turn", "mean"),
        cash_need_mean=("cash_need", "mean"),
        cash_need_min=("cash_need", "min"),
        cash_need_max=("cash_need", "max"),
    )
    .sort_values("days_with_rows", ascending=False)
)

cashdesk_stats.head(20)

In [ ]:
test_cashdesk_name = cashdesk_stats.index[0]
print(f"Тестовая касса: {test_cashdesk_name}")

test_cashdesk_df = daily_df[daily_df["atdtmco_cashdesk_name"] == test_cashdesk_name].copy()

saldo_test_forecast, need_test_forecast, need_test_scenarios, need_test_error_blocks = forecast_single_cashdesk(
    cashdesk_name=test_cashdesk_name,
    cashdesk_df=test_cashdesk_df,
    config=SARIMA_CONFIG,
)

print("Прогноз saldo_turn:")
display(saldo_test_forecast)

print("Прогноз cash_need с bootstrap p85:")
display(need_test_forecast)

print("Размер матрицы bootstrap-сценариев:", need_test_scenarios.shape)
print("Размер матрицы исторических блоков ошибок:", need_test_error_blocks.shape)

## 7. Запуск по всем кассам

In [ ]:
MAX_CASHDESKS_FOR_TEST = None

saldo_forecasts = []
need_forecasts = []
failed_cashdesks = []

cashdesk_groups = list(daily_df.groupby("atdtmco_cashdesk_name", sort=True))
if MAX_CASHDESKS_FOR_TEST is not None:
    cashdesk_groups = cashdesk_groups[:MAX_CASHDESKS_FOR_TEST]

for idx, (cashdesk_name, cashdesk_df) in enumerate(cashdesk_groups, start=1):
    print(f"[{idx}/{len(cashdesk_groups)}] Считаю кассу: {cashdesk_name}")

    try:
        saldo_forecast, need_forecast, _, _ = forecast_single_cashdesk(
            cashdesk_name=cashdesk_name,
            cashdesk_df=cashdesk_df.copy(),
            config=SARIMA_CONFIG,
        )
        saldo_forecasts.append(saldo_forecast)
        need_forecasts.append(need_forecast)
    except Exception as exc:
        failed_cashdesks.append(
            {
                "cashdesk_name": cashdesk_name,
                "error": str(exc),
                "rows": len(cashdesk_df),
                "date_min": cashdesk_df["calday"].min(),
                "date_max": cashdesk_df["calday"].max(),
            }
        )
        print(f"  Пропускаю кассу из-за ошибки: {exc}")

saldo_forecasts_df = pd.concat(saldo_forecasts, ignore_index=True) if saldo_forecasts else pd.DataFrame()
need_forecasts_df = pd.concat(need_forecasts, ignore_index=True) if need_forecasts else pd.DataFrame()
failed_cashdesks_df = pd.DataFrame(failed_cashdesks)

print("Готово")
print("saldo_forecasts_df:", saldo_forecasts_df.shape)
print("need_forecasts_df:", need_forecasts_df.shape)
print("failed_cashdesks_df:", failed_cashdesks_df.shape)

## 8. Рекомендованная потребность

In [ ]:
current_balance_df = (
    daily_df
    .sort_values(["atdtmco_cashdesk_name", "calday"])
    .groupby("atdtmco_cashdesk_name", as_index=False)
    .tail(1)
    [["atdtmco_cashdesk_name", "calday", "ns_min", "cash_need"]]
    .rename(
        columns={
            "atdtmco_cashdesk_name": "cashdesk_name",
            "calday": "current_balance_date",
            "ns_min": "current_ns_min",
            "cash_need": "current_balance_proxy",
        }
    )
)

need_recommendation_df = need_forecasts_df.merge(
    current_balance_df,
    on="cashdesk_name",
    how="left",
)

need_recommendation_df["recommended_replenishment"] = np.maximum(
    0,
    need_recommendation_df["forecast_p85"] - need_recommendation_df["current_balance_proxy"],
)

horizon_recommendation_df = (
    need_recommendation_df
    .groupby("cashdesk_name", as_index=False)
    .agg(
        forecast_start=("forecast_date", "min"),
        forecast_end=("forecast_date", "max"),
        max_p85_need=("forecast_p85", "max"),
        current_balance_proxy=("current_balance_proxy", "first"),
        current_balance_date=("current_balance_date", "first"),
        error_blocks_used=("error_blocks_used", "first"),
    )
)

horizon_recommendation_df["recommended_replenishment_for_horizon"] = np.maximum(
    0,
    horizon_recommendation_df["max_p85_need"] - horizon_recommendation_df["current_balance_proxy"],
)

horizon_recommendation_df.head()

## 9. Сохранение результатов

In [ ]:
output_dir = Path("data/forecast_results")
output_dir.mkdir(parents=True, exist_ok=True)

saldo_forecasts_df.to_csv(output_dir / "saldo_forecasts.csv", index=False)
need_forecasts_df.to_csv(output_dir / "need_forecasts_p85.csv", index=False)
need_recommendation_df.to_csv(output_dir / "need_recommendation_by_day.csv", index=False)
horizon_recommendation_df.to_csv(output_dir / "horizon_recommendation.csv", index=False)
failed_cashdesks_df.to_csv(output_dir / "failed_cashdesks.csv", index=False)

print(f"Результаты сохранены в: {output_dir.resolve()}")

## 10. Описание результатов

См. `README_cashdesk_sarima_bootstrap.md`.